# Taller Práctico: Reinforcement Learning
## Q-Learning, DQN y PPO

Bienvenido al taller. Este notebook es un template preparado para que puedas experimentar con los conceptos de Reinforcement Learning sin tener que escribir la lógica compleja desde cero. Tu objetivo principal será **ajustar hiperparámetros** y **analizar el comportamiento** de los agentes entrenados.

### Instrucciones de Preparación (Si corres en local)

**1. Crear y activar entorno virtual:**
* Windows: `python -m venv rl_env` -> `rl_env\Scripts\activate`
* macOS/Linux: `python3 -m venv rl_env` -> `source rl_env/bin/activate`

**2. Instalar dependencias:**
```bash
pip install gymnasium[toy-text,box2d] torch numpy matplotlib imageio pillow
```

In [45]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
import matplotlib.pyplot as plt
import imageio
from PIL import Image
from collections import deque
import random
import os

# Crear directorio para guardar resultados y visualizaciones
os.makedirs('resultados', exist_ok=True)

print("Dependencias cargadas correctamente.")

Dependencias cargadas correctamente.


---
## Parte 1: Q-Learning Tabular en Frozen Lake

En este entorno (`FrozenLake-v1`), el agente debe aprender a caminar sobre un lago congelado para llegar a la meta sin caer en los hoyos. Las acciones son discretas (arriba, abajo, izquierda, derecha) y el espacio de estados también.

**Tu Reto:**
Modifica los valores de `alpha`, `gamma` y `epsilon` para lograr que el agente alcance la meta de forma confiable.

In [20]:
# ==========================================
# HIPERPARÁMETROS Q-LEARNING
# ==========================================
alpha = 0.8           # Tasa de aprendizaje (0 a 1)
gamma = 0.95          # Factor de descuento (0 a 1)
epsilon = 1.0         # Exploración inicial (1 = 100% aleatorio)
epsilon_decay = 0.995 # Decaimiento de epsilon por episodio
min_epsilon = 0.01    # Exploración mínima
episodes = 2000       # Número total de intentos
# ==========================================

env_ql = gym.make("FrozenLake-v1", is_slippery=True)
Q = np.zeros((env_ql.observation_space.n, env_ql.action_space.n))
rewards_q = []

print("Entrenando Q-Learning...")
for episode in range(episodes):
    s, _ = env_ql.reset()
    done = False
    ep_reward = 0
    
    while not done:
        # Acción Epsilon-Greedy
        if np.random.rand() < epsilon:
            a = env_ql.action_space.sample()
        else:
            a = np.argmax(Q[s])
            
        s2, r, terminated, truncated, _ = env_ql.step(a)
        done = terminated or truncated
        
        # Ecuación de Actualización de Bellman
        Q[s, a] = Q[s, a] + alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
        
        s = s2
        ep_reward += r
        
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_q.append(ep_reward)

print("Entrenamiento completado.")

Entrenando Q-Learning...
Entrenamiento completado.


In [ ]:
policy = np.argmax(Q, axis=1)
arrows = {0:"←", 1:"↓", 2:"→", 3:"↑"}

for row in range(4):
    print(" ".join(arrows[policy[row*4 + col]] for col in range(4))) 

In [26]:
# ==========================================
# VISUALIZACIÓN Q-LEARNING
# ==========================================
def generar_gif_qlearning(Q, filename="resultados/frozen_lake.gif"):
    env_render = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
    frames = []
    for _ in range(5):
        s, _ = env_render.reset()
        done = False
        steps = 0
        frames.append(Image.fromarray(env_render.render()))
        while not done and steps < 50:
            a = np.argmax(Q[s]) # Ahora usamos 100% Explotación
            s, _, terminated, truncated, _ = env_render.step(a)
            done = terminated or truncated
            frames.append(Image.fromarray(env_render.render()))
            steps += 1
        env_render.close()
        
        # Pausar en el frame final para ver el resultado
        for _ in range(15): frames.append(frames[-1])
    frames[0].save(filename, save_all=True, append_images=frames[1:], duration=100, loop=0)
    print(f"Visualización generada: {filename}")

generar_gif_qlearning(Q)

Visualización generada: resultados/frozen_lake.gif


---
## Parte 2: Deep Q-Networks (DQN) en Lunar Lander

Aquí subimos de nivel. El estado del módulo lunar (`LunarLander-v2`) se compone de múltiples variables continuas (velocidad, ángulo, etc.), por lo que una tabla ya no sirve. Utilizaremos una Red Neuronal para aproximar la función Q.

**Tu Reto:**
Experimenta con el tamaño del lote de memoria (`BATCH_SIZE`), la tasa de aprendizaje (`DQN_LR`) y añade capas ocultas si lo consideras necesario.

In [34]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        # Arquitectura de la Red: Siéntete libre de añadir neuronas
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class ReplayBuffer:
    def __init__(self, buffer_size, batch_size):
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()
        rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float()
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()
        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

In [30]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        
        # Q-Network y Target Network
        self.qnetwork_local = QNetwork(state_size, action_size)
        self.qnetwork_target = QNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=5e-4)
        
        self.memory = ReplayBuffer(int(1e5), 64)
        self.t_step = 0
    
    def step(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)
        self.t_step = (self.t_step + 1) % 4 # Actualizar cada 4 steps
        if self.t_step == 0 and len(self.memory) > 64:
            experiences = self.memory.sample()
            self.learn(experiences, 0.99)

    def act(self, state, eps=0.):
        state = torch.from_numpy(state).float().unsqueeze(0)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()

        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences, gamma):
        states, actions, rewards, next_states, dones = experiences
        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))
        Q_expected = self.qnetwork_local(states).gather(1, actions)
        
        loss = F.mse_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Soft update de la red target
        tau = 1e-3
        for target_param, local_param in zip(self.qnetwork_target.parameters(), self.qnetwork_local.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)


In [35]:
def train_dqn(n_episodes, eps_start, eps_end, eps_decay):
    env = gym.make('LunarLander-v3')
    agent = DQNAgent(state_size=8, action_size=4)
    scores_window = deque(maxlen=100)
    eps = eps_start
    
    print("Iniciando entrenamiento...")
    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0
        done = False
        truncated = False
        while not (done or truncated):
            action = agent.act(state, eps)
            next_state, reward, done, truncated, _ = env.step(action)
            agent.step(state, action, reward, next_state, done or truncated)
            state = next_state
            score += reward
            
        scores_window.append(score)
        eps = max(eps_end, eps_decay*eps)
        
        print(f'\rEpisodio {i_episode}\tScore Promedio (últimos 100): {np.mean(scores_window):.2f}', end="")
        if i_episode % 100 == 0:
            print(f'\rEpisodio {i_episode}\tScore Promedio (últimos 100): {np.mean(scores_window):.2f}')
            
        if np.mean(scores_window) >= 200.0:
            print(f'\n¡Entorno resuelto en {i_episode-100} episodios!')
            break
            
    env.close()
    return agent

In [40]:
# Función de apoyo para guardar gifs (Úsala después de entrenar)
def generar_gif_dqn(agent, filepath="resultados/lunar_lander_dqn.gif"):
    env_render = gym.make('LunarLander-v3', render_mode="rgb_array")
    s, _ = env_render.reset()
    frames = []
    done = False
    
    while not done:
        frames.append(env_render.render())
        a = agent.act(s, eps=0.0)
        s, r, terminated, truncated, _ = env_render.step(a)
        done = terminated or truncated
        
    env_render.close()
    imageio.mimsave(filepath, frames, fps=30)
    print(f"GIF guardado en: {filepath}")

In [37]:
# ==========================================
# HIPERPARÁMETROS DQN
# ==========================================
DQN_EPISODES = 300
DQN_EPS_START = 1.0
DQN_EPS_DECAY = 0.995
DQN_EPS_MIN = 0.05
# ==========================================

agent_dqn = train_dqn(DQN_EPISODES, DQN_EPS_START, DQN_EPS_MIN, DQN_EPS_DECAY)

Iniciando entrenamiento...
Episodio 100	Score Promedio (últimos 100): -167.36
Episodio 200	Score Promedio (últimos 100): -105.54
Episodio 300	Score Promedio (últimos 100): -49.696


In [41]:
generar_gif_dqn(agent_dqn)

GIF guardado en: resultados/lunar_lander_dqn.gif


---
## Parte 3: Proximal Policy Optimization (PPO)

PPO es el estado del arte actual. Aquí el espacio de acciones es **continuo** (cuánta potencia exacta dar a los motores). 

**Tu Reto:**
Observa la estructura de la arquitectura del Actor-Crítico. Juega con el `eps_clip` que es el corazón de la estabilidad de PPO.

In [47]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        
        # Actor: Define la POLÍTICA (acciones continuas entre -1 y 1)
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim),
            nn.Tanh()
        )
        # Desviación estándar entrenable para la exploración
        self.actor_log_std = nn.Parameter(torch.zeros(1, action_dim))
        
        # Crítico: Define el VALOR del estado
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def act(self, state):
        action_mean = self.actor(state)
        action_std = self.actor_log_std.exp().expand_as(action_mean)
        dist = Normal(action_mean, action_std)
        
        action = dist.sample()
        action_logprob = dist.log_prob(action).sum(dim=-1)
        state_val = self.critic(state)
        
        return action.detach(), action_logprob.detach(), state_val.detach()

    def evaluate(self, state, action):
        action_mean = self.actor(state)
        action_std = self.actor_log_std.exp().expand_as(action_mean)
        dist = Normal(action_mean, action_std)
        
        action_logprobs = dist.log_prob(action).sum(dim=-1)
        dist_entropy = dist.entropy().sum(dim=-1)
        state_values = self.critic(state)
        
        return action_logprobs, torch.squeeze(state_values), dist_entropy

In [55]:
def generar_gif_ppo(agent, filepath="resultados/lunar_lander_ppo.gif"):
    env_render = gym.make('LunarLander-v3', render_mode="rgb_array", continuous=True)
    s, _ = env_render.reset()
    frames = []
    
    for _ in range(1000):
        frames.append(env_render.render())
        state_tensor = torch.FloatTensor(s).unsqueeze(0)
        with torch.no_grad():
            action, _, _ = agent.act(state_tensor)
        
        s, _, terminated, truncated, _ = env_render.step(action.numpy()[0])
        if terminated or truncated:
            for _ in range(15):
                frames.append(env_render.render())
            break
        
    env_render.close()
    imageio.mimsave(filepath, frames, fps=30)
    print(f"GIF guardado en: {filepath}")

In [51]:
def train_ppo():
    env_name = "LunarLander-v3" # Usa "LunarLanderContinuous-v2" si tienes una versión antigua de gym
    env = gym.make(env_name, continuous=True)
    
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    # Hiperparámetros de PPO
    max_episodes = 300      # Ajusta este valor según la convergencia deseada
    update_timestep = 2000   # Actualizar política cada X timesteps
    lr = 0.0003
    gamma = 0.99
    eps_clip = 0.2
    k_epochs = 40
    
    model = ActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()
    
    # Variables de memoria
    memory_states, memory_actions, memory_logprobs = [], [], []
    memory_rewards, memory_is_terminals = [], []
    
    timestep = 0
    episode_rewards = []
    
    print("Iniciando entrenamiento...")
    
    for i_episode in range(1, max_episodes + 1):
        state, _ = env.reset()
        current_ep_reward = 0
        
        for t in range(1000):
            timestep += 1
            
            # Recolectar experiencia
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                action, logprob, _ = model.act(state_tensor)
            
            memory_states.append(state_tensor)
            memory_actions.append(action)
            memory_logprobs.append(logprob)
            
            state, reward, terminated, truncated, _ = env.step(action.numpy()[0])
            done = terminated or truncated
            
            memory_rewards.append(reward)
            memory_is_terminals.append(done)
            current_ep_reward += reward
            
            # Actualización de PPO
            if timestep % update_timestep == 0:
                old_states = torch.squeeze(torch.stack(memory_states, dim=0)).detach()
                old_actions = torch.squeeze(torch.stack(memory_actions, dim=0)).detach()
                old_logprobs = torch.squeeze(torch.stack(memory_logprobs, dim=0)).detach()
                
                # Calcular retornos esperados (Monte Carlo)
                rewards = []
                discounted_reward = 0
                for r, is_terminal in zip(reversed(memory_rewards), reversed(memory_is_terminals)):
                    if is_terminal:
                        discounted_reward = 0
                    discounted_reward = r + (gamma * discounted_reward)
                    rewards.insert(0, discounted_reward)
                
                rewards = torch.tensor(rewards, dtype=torch.float32)
                rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)
                
                # Optimización de la política por K epochs
                for _ in range(k_epochs):
                    logprobs, state_values, dist_entropy = model.evaluate(old_states, old_actions)
                    advantages = rewards - state_values.detach()
                    
                    ratios = torch.exp(logprobs - old_logprobs)
                    
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1 - eps_clip, 1 + eps_clip) * advantages
                    
                    loss = -torch.min(surr1, surr2) + 0.5 * mse_loss(state_values, rewards) - 0.01 * dist_entropy
                    
                    optimizer.zero_grad()
                    loss.mean().backward()
                    optimizer.step()
                
                # Limpiar memoria
                memory_states.clear()
                memory_actions.clear()
                memory_logprobs.clear()
                memory_rewards.clear()
                memory_is_terminals.clear()
                
            if done:
                break
                
        episode_rewards.append(current_ep_reward)
        
        if i_episode % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])
            print(f"Episodio: {i_episode} \t Recompensa Media (últimos 100): {avg_reward:.2f}")
            
    env.close()
    return model

In [52]:
agent_ppo = train_ppo()

Iniciando entrenamiento...
Episodio: 100 	 Recompensa Media (últimos 100): -161.72
Episodio: 200 	 Recompensa Media (últimos 100): -35.66
Episodio: 300 	 Recompensa Media (últimos 100): 17.12


In [56]:
generar_gif_ppo(agent_ppo)

GIF guardado en: resultados/lunar_lander_ppo.gif
